# 03. Climate-change adjustment of design rainfall

This notebook extracts the annual maximum daily rainfall of 13 bias-corrected CMIP6 global climate models (Mishra et al. 2020) at the grid cell nearest the study area, forms the year-by-year ensemble-mean series for the historical reference period and for each future case, and applies the delta-change method to the observed design rainfall. It supports Sections 2.3.1 and 2.3.2 of the manuscript and produces the climate-adjusted design rainfall table.

Rainfall on a 0.25° grid is lower than point rainfall at a gauge, so the projections are used only to measure relative change (Fowler et al. 2007). For each return period $T$,

$$P_{\mathrm{fut}}(T) = P_{\mathrm{obs}}(T)\,\frac{P_{\mathrm{mod,fut}}(T)}{P_{\mathrm{mod,hist}}(T)}$$

where $P_{\mathrm{obs}}$ is the observed design depth and $P_{\mathrm{mod,hist}}$ and $P_{\mathrm{mod,fut}}$ are the GEV quantiles of the historical and future ensemble-mean series. Their ratio is the change factor.

## Configuration

In [ ]:
import glob
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path("..")
DATA_ROOT = ROOT / "data" / "cmip6"            # holds one folder per GCM
OBSERVED_DESIGN = ROOT / "results" / "02_frequency" / "design_rainfall_observed.csv"
OUTDIR = ROOT / "results" / "03_cmip6"

TARGET_LAT, TARGET_LON = 22.79, 89.57          # centroid of Wards 27, 28, 30 and 31
HIST_START, HIST_END = 1985, 2014              # historical reference period
FUTURE_WINDOWS = [(2021, 2050), (2051, 2080)]
SCENARIOS = ["ssp245", "ssp585"]
RETURN_PERIODS = [2, 5, 10, 25, 50, 100]
DIST_COL = "GEV (mm)"                          # distribution selected in notebook 02

OUTDIR.mkdir(parents=True, exist_ok=True)

## 1. Reader

Each `PrecipData` file has two header rows, holding the longitude and the latitude of every grid cell after three placeholder columns, followed by one row per day with the year, month, day and one precipitation value (mm) per cell. The reader returns the daily series of the cell nearest the target point by great-circle (haversine) distance.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between two lat/lon points."""
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1); dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dlmb/2)**2
    return 2 * R * math.asin(math.sqrt(a))


def read_precipdata(path, tlat, tlon, verbose=False):
    """Read a PrecipData file of Mishra et al. (2020) and return the daily
    rainfall series of the grid cell nearest the target point."""
    with open(path) as f:
        rows = [ln.split() for ln in f if ln.strip()]
    lons = [float(v) for v in rows[0][3:]]     # columns 1 to 3 are year, month, day
    lats = [float(v) for v in rows[1][3:]]

    dists = [haversine_km(tlat, tlon, la, lo) for la, lo in zip(lats, lons)]
    j = int(np.argmin(dists))
    if verbose:
        print(f"      nearest cell: lon {lons[j]}, lat {lats[j]} "
              f"({dists[j]:.1f} km away), out of {len(lons)} cells")

    dates, vals = [], []
    for r in rows[2:]:
        dates.append(pd.Timestamp(int(r[0]), int(r[1]), int(r[2])))
        vals.append(float(r[3 + j]))
    return pd.Series(vals, index=pd.DatetimeIndex(dates))

## 2. Annual maxima of one scenario and window

For every GCM the function keeps the requested years, takes the maximum of each calendar year and returns the per-model table together with the ensemble-mean series (the mean of the 13 annual maxima of each year). Annual maxima below 40 mm or above 900 mm are flagged for inspection.

In [ ]:
def extract_scenario(data_root, scenario, tlat, tlon, y0, y1, verbose=True):
    pattern = os.path.join(data_root, "*", scenario, "PrecipData*")
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(
            f"No PrecipData found for scenario '{scenario}'.\n"
            f"Pattern tried: {pattern}\n"
            f"Expected layout: <DATA_ROOT>/<MODEL>/{scenario}/PrecipData")

    if verbose:
        print(f"[{scenario}] found {len(files)} model files, window {y0}-{y1}")
    per_model = {}
    for path in files:
        model = path.split(os.sep)[-3]
        s = read_precipdata(path, tlat, tlon)
        s = s[(s.index.year >= y0) & (s.index.year <= y1)]
        if len(s) == 0:
            print(f"    {model}: no data in window, skipped")
            continue
        ams = s.groupby(s.index.year).max()
        flag = ""
        if ams.max() < 40 or ams.max() > 900:
            flag = "  <-- check: outside 40 to 900 mm"
        per_model[model] = ams
        if verbose:
            print(f"    {model:<16s} maxima {ams.min():.0f}-{ams.max():.0f} mm, "
                  f"mean {ams.mean():.0f}, std {ams.std():.0f}{flag}")

    table = pd.DataFrame(per_model)
    table.index.name = "Year"
    ensemble = pd.DataFrame({"Year": table.index,
                             "Rainfall_mm": table.mean(axis=1).round(2).values})
    return table, ensemble

# grid cell selected for the first model
_first = sorted(glob.glob(os.path.join(DATA_ROOT, "*", "historical", "PrecipData*")))
if not _first:
    raise FileNotFoundError(f"No historical PrecipData files under {DATA_ROOT}")
_ = read_precipdata(_first[0], TARGET_LAT, TARGET_LON, verbose=True)

## 3. Historical reference period, 1985 to 2014

In [ ]:
hist_table, hist_ens = extract_scenario(
    DATA_ROOT, "historical", TARGET_LAT, TARGET_LON, HIST_START, HIST_END)

hist_table.to_csv(OUTDIR / f"AMS_per_model_historical_{HIST_START}_{HIST_END}.csv")
hist_ens.to_csv(OUTDIR / f"AMS_ensemble_historical_{HIST_START}_{HIST_END}.csv", index=False)

print("\nEnsemble-mean annual maxima, historical period:")
print(hist_ens["Rainfall_mm"].describe())

## 4. GEV quantiles of the ensemble-mean series

The GEV distribution is fitted by L-moments, exactly as for the observed series in notebook 02.

In [ ]:
def lmoments(x):
    x = np.sort(np.asarray(x, float)); n = len(x); j = np.arange(1, n + 1)
    b0 = x.mean()
    b1 = np.sum((j - 1) / (n - 1) * x) / n
    b2 = np.sum((j - 1) * (j - 2) / ((n - 1) * (n - 2)) * x) / n
    l1 = b0; l2 = 2*b1 - b0; t3 = (6*b2 - 6*b1 + b0)/l2
    return l1, l2, t3

def gev_quantiles(x, return_periods):
    """Fit GEV by L-moments, return design depth for each return period."""
    l1, l2, t3 = lmoments(x)
    cc = 2.0/(3.0+t3) - math.log(2)/math.log(3)
    k = 7.8590*cc + 2.9554*cc**2
    k = k if abs(k) > 1e-6 else 1e-6
    gk = math.gamma(1+k)
    scale = l2*k/((1-2**(-k))*gk)
    loc = l1 - scale*(1-gk)/k
    out = {}
    for T in return_periods:
        out[T] = float(stats.genextreme.ppf(1 - 1.0/T, c=k, loc=loc, scale=scale))
    return out

hist_design = gev_quantiles(hist_ens["Rainfall_mm"].values, RETURN_PERIODS)

## 5. Change factors and future design rainfall

The extraction, the GEV fit and the delta-change calculation are repeated for the two pathways (SSP2-4.5 and SSP5-8.5) and the two future windows (2021 to 2050 and 2051 to 2080).

In [ ]:
obs = pd.read_csv(OBSERVED_DESIGN)

details = []
for scenario in SCENARIOS:
    for y0, y1 in FUTURE_WINDOWS:
        case = f"{scenario}_{y0}_{y1}"
        fut_table, fut_ens = extract_scenario(
            DATA_ROOT, scenario, TARGET_LAT, TARGET_LON, y0, y1)
        fut_table.to_csv(OUTDIR / f"AMS_per_model_{case}.csv")
        fut_ens.to_csv(OUTDIR / f"AMS_ensemble_{case}.csv", index=False)

        fut_design = gev_quantiles(fut_ens["Rainfall_mm"].values, RETURN_PERIODS)

        rows = []
        for _, r in obs.iterrows():
            T = int(r["Return period (yr)"])
            if T not in fut_design:
                continue
            cf = fut_design[T] / hist_design[T]
            obs_val = r[DIST_COL]
            rows.append({
                "Case": case,
                "Return period (yr)": T,
                "Observed (mm)": round(obs_val, 1),
                "Model historical (mm)": round(hist_design[T], 1),
                "Model future (mm)": round(fut_design[T], 1),
                "Change factor": round(cf, 3),
                "Change (%)": round((cf - 1) * 100, 1),
                DIST_COL: round(obs_val * cf, 1),          # future design depth
            })
        delta = pd.DataFrame(rows)
        details.append(delta)

        # design-rainfall table in the layout read by notebooks 04 and 05
        delta_out = delta[["Return period (yr)", DIST_COL]].copy()
        delta_out["Exceedance prob (%)"] = (100.0 / delta["Return period (yr)"]).round(2)
        delta_out = delta_out[["Return period (yr)", "Exceedance prob (%)", DIST_COL]]
        delta_out.to_csv(OUTDIR / f"design_rainfall_{case}.csv", index=False)
        print()

change_factors = pd.concat(details, ignore_index=True)
change_factors.to_csv(OUTDIR / "change_factors.csv", index=False)
change_factors.pivot(index="Return period (yr)", columns="Case", values="Change factor")

## 6. Observed and climate-adjusted design rainfall

In [ ]:
table = obs[["Return period (yr)", "Exceedance prob (%)", DIST_COL]].rename(columns={DIST_COL: "observed (mm)"})
for case, d in change_factors.groupby("Case", sort=False):
    d = d.set_index("Return period (yr)")
    table[f"{case} (mm)"] = table["Return period (yr)"].map(d[DIST_COL])
    table[f"{case} (change %)"] = table["Return period (yr)"].map(d["Change (%)"])
table.to_csv(OUTDIR / "design_rainfall_all_cases.csv", index=False)
table